# Step 7: feature engineering

**Code version:** `main` merged into this branch (PR #11, "Mark AED results in step 7 instead of dropping clips").

**What Step 7 does:** combines the Step 3 (BirdNET), 4 (acoustic), 5 (metadata) and 6 (weather) outputs into one row per clip, adds the sentinel species flags and the engineered features, and marks every clip with Step 2's AED result. No clip is dropped.

**Prediction (from reading the code, before running):**
- 1,747 rows, and 36 columns: 29 features, the 5 reference columns (clip_name, species, Recorder, Timestamp, Datetime) and the 2 AED columns (not_meaningful_prob, predicted_label).
- No blanks in the AED columns, because every clip has a Step 2 row. All labels are "meaningful" on this recording.
- The AED values match Step 2's output exactly, and the 29 feature values match the run before PR #11.
- The AED columns are in none of the HP feature lists (29, 27, 25, 23).
- Only 3 of the 10 sentinel flags are non-zero (58, 6 and 2 clips). `Eerie_Silence` is 1 for 1,681 clips.
- A clip with no AED row is kept, with blank AED columns and a warning.

**Outcome:** the prediction held. The one change is how "identical" is stated: the 29 feature values match the earlier run to within 1e-12 (rounding differences of about 1e-16 in 3 columns), not bit for bit.

In [1]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import sys, ast, subprocess
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = KWF_TEST_DIR / "outputs"
MODELS_DIR  = PIPELINE_DIR / "models"
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

import feature_engineering as fe
from hp_features import HP_FEATURES_29, HP_FEATURES_27, HP_FEATURES_25, HP_FEATURES_23

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=PIPELINE_DIR,
                        capture_output=True, text=True).stdout.strip()
print("Code version:", commit)
print("HP feature lists:", {n: len(l) for n, l in [(29, HP_FEATURES_29), (27, HP_FEATURES_27), (25, HP_FEATURES_25), (23, HP_FEATURES_23)]})

Code version: 6c5be36
HP feature lists: {29: 29, 27: 27, 25: 25, 23: 23}


**Result:** the code is `main` at 52835f3 (PR #11 included), merged into this branch as 6c5be36. The four feature lists have 29, 27, 25 and 23 names. Qian says the current HP candidate uses the 23-feature list.

**What this does:** checks each of the 10 sentinel species against the BirdNET labels file and the regional allow-list, and counts its detections on this recording.
**How to read it:** `in labels` or `in allow-list` False means that species can never be detected, so its flag would always be 0.

In [2]:
labels = [l.strip() for l in open(MODELS_DIR / "birdnet_fp16_pruned493.labels.txt")]
allow  = {l.strip() for l in open(MODELS_DIR / "geomodel_allowlist_218.txt") if l.strip()}
bird   = pd.read_csv(OUTPUTS_DIR / "nb03_birdnet_output.csv")
detected = pd.Series([s for lst in bird["species"].apply(ast.literal_eval) for s in lst]).value_counts()

rows = [{"sentinel": s, "in labels": s in labels, "in allow-list": s in allow,
         "detections (recorder 5)": int(detected.get(s, 0))} for s in fe.SENTINEL_SPECIES]
print(pd.DataFrame(rows).to_string(index=False))

                                      sentinel  in labels  in allow-list  detections (recorder 5)
    Myiothlypis fulvicauda_Buff-rumped Warbler       True           True                       58
Habia atrimaxillaris_Black-cheeked Ant-Tanager       True           True                        6
  Thamnophilus bridgesi_Black-hooded Antshrike       True           True                        0
                   Tinamus major_Great Tinamou       True           True                        0
  Patagioenas nigrirostris_Short-billed Pigeon       True           True                        0
    Ramphastos ambiguus_Yellow-throated Toucan       True           True                        0
      Cyanoloxia cyanoides_Blue-black Grosbeak       True           True                        2
                 Lipaugus unirufus_Rufous Piha       True           True                        0
      Threnetes ruckeri_Band-tailed Barbthroat       True           True                        0
                    

**Result:** all 10 sentinels are in the labels file and the allow-list, so every flag can fire. Only 3 are detected on this recording: Buff-rumped Warbler (58 clips), Black-cheeked Ant-Tanager (6) and Blue-black Grosbeak (2).

**What this does:** runs `run_feature_engineering` with the same inputs as `run_pipeline.py`: the Step 4, 3, 5 and 6 outputs, plus Step 2's CSV as `aed_csv`. It also loads `reference_features_before_pr11.csv`, a saved copy of the Nov 5 pipeline run (before PR #11), to compare against in the next cell.
**How to read it:** the log lines come from the code. "Marked N clips with AED results" shows how many clips got a label. A crash here would mean Step 7 can't run on these inputs.

In [3]:
aed_csv  = OUTPUTS_DIR / "nb02_recorder5_rerun_predictions.csv"
old_path = KWF_TEST_DIR / "reference_features_before_pr11.csv"   # Nov 5 pipeline run, before PR #11
old = pd.read_csv(old_path) if old_path.exists() else None

feats_asis = fe.run_feature_engineering(
    acoustic_csv=str(OUTPUTS_DIR / "nb04_recorder5_acoustic_features.csv"),
    birdnet_csv=str(OUTPUTS_DIR / "nb03_birdnet_output.csv"),
    metadata_csv=str(OUTPUTS_DIR / "nb05_metadata.csv"),
    weather_csv=str(OUTPUTS_DIR / "nb06_weather_matches.csv"),
    output_csv=str(OUTPUTS_DIR / "nb07_features_asis.csv"),
    aed_csv=str(aed_csv),
)

feats_asis = fe.run_feature_engineering(
    acoustic_csv=str(OUTPUTS_DIR / "nb04_recorder5_acoustic_features.csv"),
    birdnet_csv=str(OUTPUTS_DIR / "nb03_birdnet_output.csv"),
    metadata_csv=str(OUTPUTS_DIR / "nb05_metadata.csv"),
    weather_csv=str(OUTPUTS_DIR / "nb06_weather_matches.csv"),
    output_csv=str(OUTPUTS_DIR / "nb07_features_asis.csv"),
    aed_csv=str(aed_csv),
)

Loading inputs...
Merging inputs...
Encoding sentinel species...
Parsing confidence...
Engineering Patrick's features...
Marking AED results...
Marked 1,747 clips with AED results: {'meaningful': 1747}
Saving output...
Done. 1,747 rows saved to /Users/huyguy/kwf_test/outputs/nb07_features_asis.csv
Loading inputs...
Merging inputs...
Encoding sentinel species...
Parsing confidence...
Engineering Patrick's features...
Marking AED results...
Marked 1,747 clips with AED results: {'meaningful': 1747}
Saving output...
Done. 1,747 rows saved to /Users/huyguy/kwf_test/outputs/nb07_features_asis.csv


**Result:** the run finished. The log shows "Marked 1,747 clips with AED results: {'meaningful': 1747}" and 1,747 rows saved.

**What this does:** checks the new `features.csv`: the shape, the columns beyond the 29 features, blanks, the AED columns against Step 2's file, the sentinel flags, `Eerie_Silence` and `Volume_Spike_15s`. It also compares the 29 feature values with a saved copy of the Nov 5 pipeline run, which used the code before PR #11.
**How to read it:** 1,747 rows means no clip was dropped. Blank AED values, or a label or probability that differs from Step 2, would mean the join went wrong. A difference of about 1e-16 is floating-point rounding. A larger difference would mean the new code changed a feature.

In [4]:
import numpy as np

feats = feats_asis.copy()
aed = pd.read_csv(aed_csv)

print("Shape:", feats.shape, "| clips in the Step 2 output:", len(aed))
print("Columns beyond the 29 features:", [c for c in feats.columns if c not in HP_FEATURES_29])
print("Blank values in the 29 features:", feats[HP_FEATURES_29].isna().sum().sum())
print("Blank AED values:", feats[["not_meaningful_prob", "predicted_label"]].isna().sum().to_dict())
print("Labels:", feats["predicted_label"].value_counts().to_dict())

a = aed.assign(clip_name=aed["clip_name"].str.replace(".wav", "", regex=False)).set_index("clip_name")
f = feats.set_index("clip_name")
print("Label identical to Step 2:", f["predicted_label"].equals(a["predicted_label"].reindex(f.index)))
print("Max probability difference vs Step 2:", float((f["not_meaningful_prob"] - a["not_meaningful_prob"].reindex(f.index)).abs().max()))

if old is not None:
    o = old.sort_values("clip_name").reset_index(drop=True)
    n = feats.sort_values("clip_name").reset_index(drop=True)
    diff = float((o[HP_FEATURES_29] - n[HP_FEATURES_29]).abs().max().max())
    print("Largest difference in the 29 feature values vs the run before PR #11:", diff)
    print("Identical to within 1e-12:", bool(np.allclose(o[HP_FEATURES_29], n[HP_FEATURES_29], rtol=0, atol=1e-12)))

s = feats[fe.SENTINEL_SPECIES].sum()
print("\nSentinel flag sums (non-zero only):\n", s[s > 0].to_string())
print("Eerie_Silence = 1:", int(feats["Eerie_Silence"].sum()), "of", len(feats))
print("Volume_Spike_15s:", feats["Volume_Spike_15s"].describe()[["min", "mean", "max"]].round(4).to_dict())
print("First clip spike:", feats["Volume_Spike_15s"].iloc[0], "| sorted by time:", feats["Datetime"].is_monotonic_increasing)
print("Example clip_name:", feats["clip_name"].iloc[0])

Shape: (1747, 36) | clips in the Step 2 output: 1747
Columns beyond the 29 features: ['clip_name', 'species', 'Recorder', 'Timestamp', 'Datetime', 'not_meaningful_prob', 'predicted_label']
Blank values in the 29 features: 0
Blank AED values: {'not_meaningful_prob': 0, 'predicted_label': 0}
Labels: {'meaningful': 1747}
Label identical to Step 2: True
Max probability difference vs Step 2: 0.0
Largest difference in the 29 feature values vs the run before PR #11: 1.1102230246251565e-16
Identical to within 1e-12: True

Sentinel flag sums (non-zero only):
 Myiothlypis fulvicauda_Buff-rumped Warbler        58
Habia atrimaxillaris_Black-cheeked Ant-Tanager     6
Cyanoloxia cyanoides_Blue-black Grosbeak           2
Eerie_Silence = 1: 1681 of 1747
Volume_Spike_15s: {'min': 0.0, 'mean': 0.0016, 'max': 0.2493}
First clip spike: 0.0 | sorted by time: True
Example clip_name: Audio_Moth_5_20250319_163237


**Result:** 1,747 rows and 36 columns. The 7 columns beyond the 29 features are clip_name, species, Recorder, Timestamp, Datetime, not_meaningful_prob and predicted_label. There are 0 blanks in the 29 features and 0 in the AED columns. All labels are "meaningful", identical to Step 2's file (probability difference 0.0). The 29 feature values match the Nov 5 pipeline run (before PR #11) to within 1e-12: the largest difference is about 1e-16, in 3 of 29 columns (hour_sin, Volume_Spike_15s, Volume_Wind_Ratio), which is floating-point rounding. Sentinel flags: 58, 6 and 2 clips. `Eerie_Silence` = 1 for 1,681 of 1,747 clips. The first clip's spike is 0.0 (maximum 0.2493), and rows are sorted by time. `clip_name` has no `.wav`, because Step 7 strips it.

**What this does:** checks `features.csv` against each of the team's four feature lists, checks that the AED columns are not in any list, and asks `select_features` to pick each list in the team's order.
**How to read it:** "missing: none" means every list can be selected. The column order in `features.csv` differs from the team's lists, but `select_features` returns the columns in the team's order, so the order in the CSV doesn't matter when the model code uses it.

In [5]:
from hp_features import select_features

lists = [(29, HP_FEATURES_29), (27, HP_FEATURES_27), (25, HP_FEATURES_25), (23, HP_FEATURES_23)]
for n, lst in lists:
    print(f"HP_FEATURES_{n}: missing from features.csv:", [c for c in lst if c not in feats.columns] or "none",
          "| select_features order correct:", list(select_features(feats, lst).columns) == list(lst))

print("AED columns inside any HP list:",
      [c for c in ["not_meaningful_prob", "predicted_label"] if any(c in lst for _, lst in lists)] or "none")
print("features.csv column order equals the team's 29 list:", [c for c in feats.columns if c in HP_FEATURES_29] == list(HP_FEATURES_29))

HP_FEATURES_29: missing from features.csv: none | select_features order correct: True
HP_FEATURES_27: missing from features.csv: none | select_features order correct: True
HP_FEATURES_25: missing from features.csv: none | select_features order correct: True
HP_FEATURES_23: missing from features.csv: none | select_features order correct: True
AED columns inside any HP list: none
features.csv column order equals the team's 29 list: False


**Result:** all four lists (29, 27, 25 and 23) are complete in `features.csv`, and `select_features` returns each one in the team's order. The AED columns are in none of the lists, so they are marking only, not model features. The column order in `features.csv` differs from the team's 29 list, which is harmless when the model code picks features with `select_features`.

**What this does:** removes every 100th clip (18 clips) from Step 2's CSV, which simulates clips AED couldn't read, and runs Step 7 again.
**How to read it:** all 1,747 rows should be kept, with blank AED columns for the 18 clips and a warning printed. The old code dropped clips. This replaces that filter test. This is simulated, because every real clip has an AED row on this recording.

In [6]:
aed.drop(index=aed.index[::100]).to_csv(OUTPUTS_DIR / "nb07_aed_missing_rows.csv", index=False)

feats_m = fe.run_feature_engineering(
    acoustic_csv=str(OUTPUTS_DIR / "nb04_recorder5_acoustic_features.csv"),
    birdnet_csv=str(OUTPUTS_DIR / "nb03_birdnet_output.csv"),
    metadata_csv=str(OUTPUTS_DIR / "nb05_metadata.csv"),
    weather_csv=str(OUTPUTS_DIR / "nb06_weather_matches.csv"),
    output_csv=str(OUTPUTS_DIR / "nb07_features_missing_aed.csv"),
    aed_csv=str(OUTPUTS_DIR / "nb07_aed_missing_rows.csv"),
)
print("\nRows kept:", len(feats_m), "| clips with blank AED columns:", int(feats_m["predicted_label"].isna().sum()))

Loading inputs...
Merging inputs...
Encoding sentinel species...
Parsing confidence...
Engineering Patrick's features...
Marking AED results...
Marked 1,747 clips with AED results: {'meaningful': 1729, nan: 18}
Saving output...
Done. 1,747 rows saved to /Users/huyguy/kwf_test/outputs/nb07_features_missing_aed.csv

Rows kept: 1747 | clips with blank AED columns: 18


**Result:** all 1,747 rows were kept. 18 clips have blank AED columns, and the warning "18 of 1,747 clips have no AED result" was printed. No clip is dropped. This was simulated, because every clip on this recording has an AED row.

## Observations (Step 7, main at 52835f3)

**Tested:** `run_feature_engineering` on recorder 5 (1,747 clips) with the Step 3, 4, 5 and 6 outputs and Step 2's CSV as `aed_csv`, the same way `run_pipeline.py` calls it.

**Result:** works. 1,747 rows and 36 columns: the 29 features, the AED columns `not_meaningful_prob` and `predicted_label`, and 5 reference columns. Every clip is kept and marked with its AED result (all "meaningful" on this recording), and the values match Step 2's output exactly. The 29 feature values match the Nov 5 pipeline run (before PR #11) to within 1e-12. The AED columns are in none of the HP feature lists (29, 27, 25, 23), and every list can be selected with `select_features`.

**Missing AED rows:** simulated (18 clips): all rows kept, with blank AED columns and a warning.

**Observations**
1. `Eerie_Silence` = 1 for 96% of clips on this recording (only 3 of 10 sentinels detected).
2. The column order in `features.csv` differs from the team's lists, so model code should pick features with `select_features`.

**Not tested:** the HP model (Step 8 is a placeholder); multiple recorders; real clips flagged `not_meaningful` (recorder 5 has none); `--no_weather` here (covered in notebook 99).

**Historical:** before PR #11, Step 7 dropped clips that Step 2 flagged `not_meaningful`. Before PR #2 and #3, it failed on lowercase weather column names, and 599 clips had blank weather.